# Modularity on the Karate Club Graph
**DSC212: Graph Theory Module, Research Assignment**

This notebook implements **recursive spectral bisection** (Newman, 2006) from scratch on Zachary's karate club network, visualises the graph after every split, tracks four NetworkX node metrics across the splits, and discusses the results.

Node IDs follow NetworkX's convention (0 to 33): node **0 is Mr. Hi** (the instructor) and node **33 is the club president** (the administrator).

## 1. Setup: graph, adjacency matrix and modularity matrix
We build $\mathbf{A}$, the degree vector $\mathbf{k}$, and $\mathbf{B}=\mathbf{A}-\frac{\mathbf{k}\mathbf{k}^\top}{2m}$, then sanity-check two facts from the handout: $\mathbf{B}$ is real symmetric (Proposition 1) and its rows sum to zero (so the all-ones vector has eigenvalue 0).

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

G = nx.karate_club_graph()
nodes = sorted(G.nodes())            # 0..33
n, m = G.number_of_nodes(), G.number_of_edges()

A = nx.to_numpy_array(G, nodelist=nodes, weight=None)   # unweighted 0/1 adjacency matrix
k = A.sum(axis=1)                           # degree vector
B = A - np.outer(k, k) / (2 * m)            # modularity matrix

print(f"n = {n} nodes, m = {m} edges, sum of degrees = {int(k.sum())} (= 2m)")
print("B symmetric:           ", np.allclose(B, B.T))
print("rows of B sum to zero: ", np.allclose(B.sum(axis=1), 0))
print("largest eigenvalue of full B:", round(np.linalg.eigvalsh(B)[-1], 4))

## 2. Recursive spectral bisection
For a community $C$ we form the restricted block $\mathbf{B}^{(C)}$ (rows and columns of $\mathbf{B}$ indexed by $C$), compute its leading eigenpair $(\lambda_1^{(C)},\mathbf{u}_1^{(C)})$ and

* if $\lambda_1^{(C)}\le 0$ the community is **indivisible**, so we stop;
* otherwise we split it by the sign of $\mathbf{u}_1^{(C)}$ ($+1$ if the entry is $>0$, else $-1$).

Communities are processed from a queue, so every split is one *iteration*. The modularity of any partition is
$Q=\frac{1}{2m}\sum_{C}\sum_{i,j\in C}B_{ij}$, which equals $\frac{1}{4m}\mathbf{s}^\top\mathbf{B}\mathbf{s}$ for two groups, and we cross-check it against NetworkX.

In [ ]:
TOL = 1e-9   # numerical zero for the eigenvalue test

def restricted_B(C):
    # Return B^(C) and the (sorted) node indices it refers to
    idx = np.array(sorted(C))
    return B[np.ix_(idx, idx)], idx

def leading_eigenpair(BC):
    # Leading eigenvalue/eigenvector of a real symmetric matrix
    vals, vecs = np.linalg.eigh(BC)          # ascending eigenvalues
    return vals[-1], vecs[:, -1]

def split_community(C):
    # Try to bisect C. Returns (C_plus, C_minus, lambda1) or None if indivisible.
    BC, idx = restricted_B(C)
    lam, u = leading_eigenpair(BC)
    if lam <= TOL:                           # stopping rule: lambda_1 <= 0
        return None
    s = np.where(u > 0, 1, -1)               # sign thresholding
    C_plus, C_minus = set(idx[s == 1].tolist()), set(idx[s == -1].tolist())
    if not C_plus or not C_minus:            # degenerate split, nothing to gain
        return None
    return C_plus, C_minus, lam

def modularity_Q(communities):
    # Q = 1/(2m) * sum over communities of sum_{i,j in C} B_ij
    total = 0.0
    for C in communities:
        idx = np.array(sorted(C))
        total += B[np.ix_(idx, idx)].sum()
    return total / (2 * m)

def recursive_bisection():
    # Run recursive bisection; return a list of per-iteration snapshots.
    # Snapshot 0 is the unsplit graph. Community ids are persistent: after a split
    # the half containing the parent's smallest node keeps the parent's id and the
    # other half gets a fresh id, so colours are stable across iterations.
    label = {v: 0 for v in nodes}
    comms = {0: set(nodes)}
    next_id = 1
    queue = [0]
    history = [dict(iteration=0, label=dict(label), communities=[set(comms[0])],
                    lam=None, split_of=None, Q=modularity_Q([comms[0]]))]
    while queue:
        cid = queue.pop(0)
        res = split_community(comms[cid])
        if res is None:
            continue                          # indivisible: leave as final community
        C_plus, C_minus, lam = res
        keep, new = (C_plus, C_minus) if min(comms[cid]) in C_plus else (C_minus, C_plus)
        comms[cid], comms[next_id] = keep, new
        for v in new:
            label[v] = next_id
        queue += [cid, next_id]
        next_id += 1
        snap = [set(c) for c in comms.values()]
        history.append(dict(iteration=len(history), label=dict(label), communities=snap,
                            lam=lam, split_of=cid, Q=modularity_Q(snap)))
    return history

history = recursive_bisection()

print(f"{len(history) - 1} splits performed -> {len(history[-1]['communities'])} final communities\n")
for h in history:
    nx_Q = nx.community.modularity(G, h["communities"], weight=None)
    sizes = sorted(len(c) for c in h["communities"])
    lam = "-" if h["lam"] is None else f"{h['lam']:.4f}"
    print(f"iter {h['iteration']}: lambda1(split)={lam:>7}  sizes={sizes}  "
          f"Q={h['Q']:.4f}  (networkx Q={nx_Q:.4f})")

In [ ]:
final = history[-1]["communities"]
for i, C in enumerate(sorted(final, key=min)):
    print(f"Community {i + 1} ({len(C)} nodes): {sorted(C)}")

# Compare with Zachary's real split (the 'club' attribute: Mr. Hi vs Officer)
truth = {v: G.nodes[v]["club"] for v in nodes}
print("\nOverlap with the real split (Mr. Hi / Officer):")
for i, C in enumerate(sorted(final, key=min)):
    hi = sum(truth[v] == "Mr. Hi" for v in C)
    print(f"  Community {i + 1}: {hi} Mr. Hi members, {len(C) - hi} Officer members")

## 3. Visualisation after each split
A single **fixed** spring layout (computed once) is used for all plots, so nodes never move between pictures; only colours change. Each community has a persistent colour, and nodes are labelled by ID.

In [ ]:
pos = nx.spring_layout(G, seed=42)              # fixed layout reused everywhere
palette = plt.get_cmap("tab10").colors

n_iter = len(history)
ncols = 3
nrows = int(np.ceil(n_iter / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5.6 * nrows))
axes = np.atleast_1d(axes).ravel()

for ax, h in zip(axes, history):
    colors = [palette[h["label"][v] % len(palette)] for v in nodes]
    nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.35, width=0.9)
    nx.draw_networkx_nodes(G, pos, ax=ax, node_color=colors, node_size=330,
                           edgecolors="black", linewidths=0.6)
    nx.draw_networkx_labels(G, pos, ax=ax, font_size=8)
    title = "Iteration 0: whole graph" if h["iteration"] == 0 else f"Iteration {h['iteration']}"
    ax.set_title(f"{title}\n{len(h['communities'])} communities, Q = {h['Q']:.3f}")
    ax.axis("off")
for ax in axes[n_iter:]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Node metrics after every split
The handout asks for four NetworkX metrics: `degree_centrality`, `betweenness_centrality`, `closeness_centrality` and `clustering`.

Computed on the whole graph $G$ these values would be identical after every split and there would be nothing to "evolve". So, after each split, each metric is computed on the **subgraph induced by the node's own community** (`G.subgraph(C)`). The metric then describes how central/clustered the node is *within its current community*. Iteration 0 is the whole graph, so it doubles as the global baseline. Nodes alone in a community get 0 for every metric.

In [ ]:
METRICS = {
    "Degree centrality":      nx.degree_centrality,
    "Betweenness centrality": nx.betweenness_centrality,
    "Closeness centrality":   nx.closeness_centrality,
    "Clustering coefficient": nx.clustering,
}

def metrics_for_partition(communities):
    # Compute every metric inside each community's induced subgraph.
    out = {name: {} for name in METRICS}
    for C in communities:
        H = G.subgraph(C)
        for name, fn in METRICS.items():
            if len(C) == 1:
                vals = {v: 0.0 for v in C}
            else:
                vals = fn(H)
            out[name].update(vals)
    return out

# dictionary keyed by iteration number, as required by the handout
metric_history = {h["iteration"]: metrics_for_partition(h["communities"]) for h in history}

import pandas as pd
summary = pd.DataFrame({
    name: pd.Series(metric_history[len(history) - 1][name]) for name in METRICS
}).round(3)
print("Metrics at the final iteration (within-community):")
summary.head(10)

## 5. Evolution of the metrics across iterations
Every node is drawn as one line. The five highest-degree nodes in the full graph are highlighted and labelled; all other nodes are shown in light grey for context.

In [ ]:
iters = sorted(metric_history)
global_deg = metric_history[0]["Degree centrality"]
highlight = sorted(nodes, key=lambda v: -global_deg[v])[:5]
hl_colors = dict(zip(highlight, palette))

fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True)
for ax, name in zip(axes.ravel(), METRICS):
    for v in nodes:
        y = [metric_history[it][name][v] for it in iters]
        if v in hl_colors:
            continue
        ax.plot(iters, y, color="lightgrey", lw=1, zorder=1)
    for v in highlight:
        y = [metric_history[it][name][v] for it in iters]
        ax.plot(iters, y, color=hl_colors[v], lw=2.4, marker="o", label=f"node {v}", zorder=3)
    ax.set_title(name)
    ax.set_xticks(iters)
    ax.set_xlabel("iteration (number of splits so far)")
    ax.set_ylabel("value")
    ax.grid(alpha=0.25)
axes[0, 0].legend(title="top-5 degree nodes", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Which nodes stay near the top of each ranking across *all* iterations?
TOP = 5
rows = []
for name in METRICS:
    counts = {v: 0 for v in nodes}
    for it in iters:
        ranked = sorted(nodes, key=lambda v: -metric_history[it][name][v])[:TOP]
        for v in ranked:
            counts[v] += 1
    best = sorted(nodes, key=lambda v: (-counts[v], v))[:TOP]
    rows.append({"Metric": name,
                 f"Nodes most often in top-{TOP} (appearances out of {len(iters)})":
                 ", ".join(f"{v} ({counts[v]})" for v in best)})
pd.DataFrame(rows).set_index("Metric")

## 6. Discussion
**What the algorithm found.** The first split (iteration 1) cuts the club into 16 and 18 members and reaches Q = 0.372. Compared with the real fission, only **node 8** lands on the wrong side, so a method that only sees the graph structure rediscovers the split that actually happened. Further splits keep raising Q, but with diminishing returns: 0.372, 0.390, 0.408, 0.409. The recursion then stops with **5 communities** (Q = 0.409) because no remaining block has a positive leading eigenvalue. The last split only peels off node 11 (a single member whose only friend is node 0) for a gain of about 0.001, so the "real" structure is the 4-group picture reached at iteration 3, with the final split being a marginal refinement.

**Which nodes stay central.** Using the top-5 ranking at every iteration (table above):

* **Nodes 0 (Mr. Hi) and 33 (the president)** are in the top 5 for degree, betweenness and closeness at all 5 iterations, and node 32 does the same for degree and betweenness. They are the hubs of the two factions, and splitting the graph does not dethrone them: inside their own community they end up adjacent to *everyone* (within-community degree and closeness reach 1.0).
* **Nodes 1 and 31** also stay near the top (degree/closeness and betweenness respectively), as lieutenants of the two hubs.
* **Clustering coefficient tells a different story.** The persistent leaders are low-degree nodes such as **7, 12, 14, 8 and 13**, which sit inside tight cliques where all of their few neighbours know each other. The hubs have only modest clustering (node 33 stays below 0.2 throughout), because they connect to several otherwise separate groups.

**How community structure shapes the metrics.**
1. *Normalisation changes with community size.* Each metric is computed inside a community and normalised by that community's size, so absolute values jump after a split (for example node 0's betweenness goes from 0.44 on the whole graph to 0.69 after the first split and 0.35 later). Rankings are the more reliable thing to compare across iterations, not raw values.
2. *Hubs become "local" hubs.* Once a community is small, its hub reaches everyone directly, so degree and closeness saturate at 1.0, as seen for nodes 0 and 33 from iteration 3 onward.
3. *Bridges lose their role.* The edges that carry the most betweenness in the full graph are the ones between factions. Cutting along community boundaries removes them, so betweenness inside a community mostly reflects how a node connects sub-groups within that community.
4. *Clustering rises when sparse cross-community edges are removed.* Node 0's clustering goes from 0.15 on the whole graph to 0.36 once the graph is reduced to its own, denser community.

**Caveat on the method.** The handout restricts the modularity matrix to a community by taking the plain block $B_{ij}$, $i,j\in C$. Newman's full generalised formulation also subtracts $\delta_{ij}\sum_{l\in C}B_{il}$ on the diagonal, which makes the stopping rule exact for the true modularity change $\Delta Q$. Here we follow the handout, and the recursion still stops cleanly, but the last splits (iterations 3 and 4) are the ones most likely to differ under the generalised version. Results also depend on the eigenvector sign rule (entries equal to 0 go to the negative group) and are deterministic for this graph.

## References
1. M. E. J. Newman, "Modularity and community structure in networks," *PNAS* 103(23):8577-8582, 2006.
2. W. W. Zachary, "An information flow model for conflict and fission in small groups," *Journal of Anthropological Research* 33(4):452-473, 1977.
3. A. Hagberg, D. Schult, P. Swart, "Exploring network structure, dynamics, and function using NetworkX," *SciPy 2008*.